In [1]:
import numpy as np
from netCDF4 import Dataset
import matplotlib.pyplot as plt
#import cartopy.crs as crs
from cartopy.io.shapereader import Reader
from cartopy.feature import ShapelyFeature
from cartopy.feature import NaturalEarthFeature
import cartopy.crs as ccrs
from datetime import datetime, timedelta
from metpy.units import units
from metpy.calc import dewpoint_from_specific_humidity, relative_humidity_from_specific_humidity, wind_speed
import pyart
import geopy.distance as distance
import haversine


## You are using the Python ARM Radar Toolkit (Py-ART), an open source
## library for working with weather radar data. Py-ART is partly
## supported by the U.S. Department of Energy as part of the Atmospheric
## Radiation Measurement (ARM) Climate Research Facility, an Office of
## Science user facility.
##
## If you use this software to prepare a publication, please cite:
##
##     JJ Helmus and SM Collis, JORS 2016, doi: 10.5334/jors.119



In [2]:
#print(P2)

In [3]:
#ncfile1 = Dataset('/glade/work/mawilson/DART_mpd/observations/utilities/threed_sphere/obs_epoch_100mIOP6.nc')
ncfile1 = Dataset('/glade/work/mawilson/DART_mpd/observations/utilities/threed_sphere/obs_epoch_100mIOP4.nc')
#ncfile1 = Dataset('/glade/work/mawilson/DART_mpd/observations/utilities/threed_sphere/obs_epoch_100mnew.nc')


In [4]:
location = ncfile1.variables['location'][:]
qc = ncfile1.variables['qc'][:]
obstype = ncfile1.variables['obs_type'][:]
obstypemd = ncfile1.variables['ObsTypesMetaData'][:]
obs_val = ncfile1.variables['observations'][:]
which_vert = ncfile1.variables['which_vert'][:]
times = ncfile1.variables['time'][:]
print(obstype[obstype==127])
qc_new = []
for i in range(len(qc)):
    qc_d = qc[i][0]
    qc_new.append(qc_d)
qc_new = np.asarray(qc_new)

[]


In [5]:
otype_s = []
obs_s = []
lon_s = []
lat_s = []
elev_s = []
error_s = []
time_s = []

#Manually set errors for different otypes
# 1.2 K T err recommended by Soyoung Ha, use same for Td initially
err_T = 1.0
err_Td = 1.0
#Try a 2 m/s wind error
err_U = 1.5
err_V = 1.5

minute_range = np.arange(180,185,5)
#minute_range = np.arange(350,365,5)
#minute_range = np.arange(595,605,5)

for mins in minute_range:
    #dt_start = datetime(2022,9,15,0,0)
    dt_start = datetime(2022,7,19,0,0)
    #dt_start = datetime(2021,6,4,0,0)
    
    dt = dt_start + timedelta(minutes=int(mins))
    dt_2 = dt - timedelta(minutes=15)
    print(dt)
    #dt = datetime(2022,9,15,9,55)
    #wrfout = Dataset('/glade/derecho/scratch/mawilson/20210604_newcase/nature_100IOP6/final_nature/wrfout_d02_2022-'+str(dt.isoformat()[5:7])+'-'+str(dt.isoformat()[8:10])+'_'+str(dt.isoformat()[11:13])+':'+str(dt.isoformat()[14:16])+':00')
    #wrfout = Dataset('/glade/derecho/scratch/mawilson/20210604_newcase/nature_100IOP4/final_nature/wrfout_d02_2022-'+str(dt.isoformat()[5:7])+'-'+str(dt.isoformat()[8:10])+'_'+str(dt.isoformat()[11:13])+':'+str(dt.isoformat()[14:16])+':00')
    #wrfout = Dataset('/glade/campaign/ral/aap/mawilson/nature_runs/JUNE/final_nature/wrfout_d02_2021-'+str(dt.isoformat()[5:7])+'-'+str(dt.isoformat()[8:10])+'_'+str(dt.isoformat()[11:13])+':'+str(dt.isoformat()[14:16])+':00')
    wrfout = Dataset('/glade/campaign/ral/aap/mawilson/nature_runs/IOP4/final_nature/wrfout_d02_2022-'+str(dt.isoformat()[5:7])+'-'+str(dt.isoformat()[8:10])+'_'+str(dt.isoformat()[11:13])+':'+str(dt.isoformat()[14:16])+':00')
    
    lon = wrfout.variables['XLONG']
    lat = wrfout.variables['XLAT']
    U10 = wrfout.variables['U10']
    V10 = wrfout.variables['U10']
    T2 = np.asarray(wrfout.variables['T2'])*units('K')
    T2F = T2 .to('degF')
    Q2 = np.asarray(wrfout.variables['Q2'])
    P2 = np.asarray(wrfout.variables['PSFC'][:]/100) * units('hPa')
    Td2 = dewpoint_from_specific_humidity(P2[0,:,:], T2[0,:,:], Q2[0,:,:]*units('kg/kg'))
    RH2 = relative_humidity_from_specific_humidity(P2[0,:,:], T2[0,:,:], Q2[0,:,:]*units('kg/kg'))
    SPD10 = wind_speed(np.asarray(U10)*units('m/s'), np.asarray(V10)*units('m/s'))
    cloud=wrfout.variables['QCLOUD']
    
    
    #otype = 107
    #otype = 105
    #otype = 106
    #otype = 108
    #otype = 142
    otype_list = [107,105,106,142]
    #otype_list = [107]
    for otype in otype_list:
        loc_T2 = location[obstype==otype, :]
        qc_T2 = qc_new[obstype==otype]
        obs_T2 = obs_val[obstype==otype, :]
        lons_T2 = loc_T2[:,0]
        lats_T2 = loc_T2[:,1]
        elev_T2 = loc_T2[:,2]
        time_T2 = times[obstype==otype]
        lons_T2[lons_T2 > 180] = lons_T2[lons_T2 > 180] - 360
        
        #Convert WRF file time into same units as the obs_seq time
        dt_tot = (dt_2 - datetime(1601,1,1)).total_seconds() / 86400
        time_diff = np.abs(dt_tot - time_T2)
        
        #Get obs within +- 15 minutes of 0245 utc
        time_T3 = time_T2[time_diff<(900/86400)]
        loc_T3 = loc_T2[time_diff<(900/86400), :]
        qc_T3 = qc_T2[time_diff<(900/86400)]
        lons_T3 = lons_T2[time_diff<(900/86400)]
        lats_T3 = lats_T2[time_diff<(900/86400)]
        elev_T3 = elev_T2[time_diff<(900/86400)]
        obs_T3 = obs_T2[time_diff<(900/86400), :]
        
        if len(obs_T3)==0:
            print('NO OBS IN WINDOW')
        for k in range(len(lons_T3)):
            latp=lats_T3[k]
            lonp=lons_T3[k]
            #Get location for each ob in model land
            lon1d = np.ndarray.flatten(lon[0,:,:])
            lat1d = np.ndarray.flatten(lat[0,:,:])
            station = []
            points = []
            for i in range(len(lon1d)):
                points.append((lat1d[i],lon1d[i]))
                station.append((latp,lonp))
            dist = haversine.haversine_vector(station,points)
            dist2=dist.reshape(lon.shape[1],lon.shape[2])
            print(lon[0,:,:][np.where(dist2==np.min(dist2))])
            print(lat[0,:,:][np.where(dist2==np.min(dist2))])
            print(np.where(dist2==np.min(dist2)))
            st_xind = np.where(dist2==np.min(dist2))[0][0]
            st_yind = np.where(dist2==np.min(dist2))[1][0]
            if otype == 107:
                T2_a = T2[0,st_xind,st_yind].magnitude
                err_fin = err_T
            elif otype == 105:
                T2_a = U10[0,st_xind,st_yind]
                err_fin = err_U
            elif otype == 106:
                T2_a = V10[0,st_xind,st_yind]
                err_fin = err_V
            elif otype == 142:
                T2_a = (Td2[st_xind,st_yind].to('K')).magnitude
                err_fin = err_Td
                otype=121
            #If you want to change the error assumption, just change the scale in this line
            error = np.random.normal(loc=0.0, scale=np.sqrt(obs_T3[k,1]))
            #6/17/2024 MW adding code to limit added error to 1.5 times the error assumtion in DART
            if np.abs(error/4) > (np.sqrt(obs_T3[k,1])*1.0):
                error = (error / np.abs(error)) * (np.sqrt(obs_T3[k,1])*1.0)
            T2_b = T2_a + (error/4)
            print(T2_a, (error/4))
            print(dt)
            
            #Append obs to arrays for writing to obs_seq file later
            otype_s.append(otype)
            obs_s.append(T2_b)
            lon_s.append(lonp)
            lat_s.append(latp)
            elev_s.append(elev_T3[k])
            error_s.append(err_fin**2)
            time_s.append(time_T3[k])

2022-07-19 03:00:00
[-84.39989]
[39.530434]
(array([862]), array([797]))
296.2795 0.5285946877935186
2022-07-19 03:00:00
[-84.25053]
[39.460365]
(array([793]), array([913]))
297.13782 -1.760354279788309
2022-07-19 03:00:00
[-85.260086]
[39.350243]
(array([679]), array([133]))
296.24115 -0.6685098415181638
2022-07-19 03:00:00
[-84.77993]
[39.24957]
(array([579]), array([505]))
296.72275 0.5090905816092526
2022-07-19 03:00:00
[-84.21958]
[39.08033]
(array([413]), array([941]))
296.8498 -0.8947351978658237
2022-07-19 03:00:00
[-84.67051]
[39.04025]
(array([370]), array([591]))
297.7771 1.4736419492132464
2022-07-19 03:00:00
[-84.42023]
[39.099823]
(array([431]), array([785]))
297.86328 0.631245321661493
2022-07-19 03:00:00
[-84.2305]
[39.600117]
(array([933]), array([927]))
295.2985 0.10669582505116612
2022-07-19 03:00:00
[-84.51945]
[39.360283]
(array([691]), array([706]))
297.27142 -0.9681557879702227
2022-07-19 03:00:00
[-84.21958]
[39.08033]
(array([413]), array([941]))
296.8498 0.088

In [6]:
#Convert lats and lons to radians for DART, because why not
lon_DART = np.radians(np.asarray(lon_s))
lat_DART = np.radians(np.asarray(lat_s))

lon_DART = np.where(lon_DART > 0.0, lon_DART, lon_DART+(2.0*np.pi))

#Convert time into DART format. This is hacky now, improve later
#day_DART = 154024
day_DART = 153966
#day_DART = 153556
seconds_DART = (np.asarray(time_s) - day_DART) * 86400

In [7]:
#Sort everything in time order
inds_time = seconds_DART.argsort()
print(seconds_DART)
print(seconds_DART[inds_time])
seconds_DART1 = seconds_DART[inds_time]
seconds_DART1[seconds_DART1 < 0] = 0
obs_s1 = np.asarray(obs_s)[inds_time]
lon_DART1 = lon_DART[inds_time]
lat_DART1 = lat_DART[inds_time]
elev_s1 = np.asarray(elev_s)[inds_time]
otype_s1 = np.asarray(otype_s)[inds_time]
error_s1 = np.asarray(error_s)[inds_time]

[ 9299.99999888  9299.99999888  9299.99999888  9299.99999888
  9299.99999888 10320.00000095 10379.99999989 10379.99999989
 10379.99999989 10500.00000028 10500.00000028 10500.00000028
 10500.00000028 10500.00000028  9299.99999888  9299.99999888
  9299.99999888  9299.99999888  9299.99999888 10320.00000095
 10379.99999989 10379.99999989 10379.99999989 10500.00000028
 10500.00000028 10500.00000028 10500.00000028 10500.00000028
  9299.99999888  9299.99999888  9299.99999888  9299.99999888
  9299.99999888 10320.00000095 10379.99999989 10379.99999989
 10379.99999989 10500.00000028 10500.00000028 10500.00000028
 10500.00000028 10500.00000028  9299.99999888  9299.99999888
  9299.99999888  9299.99999888  9299.99999888 10320.00000095
 10379.99999989 10379.99999989 10379.99999989 10500.00000028
 10500.00000028 10500.00000028 10500.00000028 10500.00000028]
[ 9299.99999888  9299.99999888  9299.99999888  9299.99999888
  9299.99999888  9299.99999888  9299.99999888  9299.99999888
  9299.99999888  9299.9

In [8]:
for bigfoot in [1,2]:
    print(bigfoot)

    #Write the simulated obs out to an obs_seq file
    #filename = 'SIM_METAR_JUNE_final03TDf'
    filename = 'SIM_METAR_IOP4_final03TDf'
    
    #filename = 'SIM_METAR_IOP4_final03'
    #filename = 'SIM_METAR_JUNE_final03'
    
    #filename = 'SIM_METAR_IOP6_finaloneob'
    fi = open(filename, "w")
    fi.write(" obs_sequence\n")
    fi.write("obs_kind_definitions\n")
    fi.write("    %d \n" %(1))
    fi.write("    %d          %s   \n" %(107, 'METAR_TEMPERATURE_2_METER'))
    fi.write("    %d          %s   \n" %(105, 'METAR_U_10_METER_WIND'))
    fi.write("    %d          %s   \n" %(106, 'METAR_V_10_METER_WIND'))
    fi.write("    %d          %s   \n" %(121, 'METAR_DEWPOINT_2_METER'))
    
    fi.write("  num_copies:            %d  num_qc:            %d\n" % (1,1))
    fi.write(" num_obs:       %d  max_num_obs:       %d\n" % (len(obs_s1), len(obs_s1)))
    fi.write("MADIS observation\n")
    fi.write("Data QC\n")
    
    fi.write("  first:            %d  last:       %d\n" % (1, len(obs_s1)))
    
    for q in range(len(obs_s1)):
    
        fi.write(" OBS            %d\n" % (q+1) )
        fi.write("   %20.14f\n" % obs_s1[q] )
        fi.write("   %20.14f\n" % 1.0 )
    
        if q+1 == 1:
            fi.write(" %d %d %d\n" % (-1, q+2, -1) ) #First obs
        elif q+1 == len(obs_s1):
            fi.write(" %d %d %d\n" % (q, -1, -1) ) #Last obs
        else:
            fi.write(" %d %d %d\n" % (q, q+2, -1) )
    
        fi.write("obdef\n")
        fi.write("loc3d\n")
        fi.write("    %20.14f          %20.14f          %20.14f     %d\n" %
                           (lon_DART1[q], lat_DART1[q], elev_s1[q], -1))
        fi.write("kind\n")
    
        fi.write("     %d     \n" % otype_s1[q])
    
        fi.write("    %d          %d     \n" % (seconds_DART1[q], day_DART))
    
        fi.write("    %20.14f  \n" % error_s1[q])

1
2


In [9]:
print(obs_T3[:,1])

#Get a randomly-generated error value to add to the synthetic ob 
#by sampling a Gaussian distribution with the same standard deviation as
#the assumed error from DART
error = np.random.normal(loc=0.0, scale=np.sqrt(obs_T3[0,1]))
print(error)

[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
-1.167533999450682


In [10]:
print(len(obs_s))

56


In [11]:
#Convert WRF file time into same units as the obs_seq time
dt_tot = (dt - datetime(1601,1,1)).total_seconds() / 86400

time_diff = np.abs(dt_tot - time_T2)

print(time_diff[time_diff<(150/86400)])

print(dt_tot)
print(time_diff)

[]
153966.125
[0.13055555554456078 0.12986111111240461 0.12986111111240461
 0.12986111111240461 0.12847222221898846 0.12847222221898846
 0.12847222221898846 0.12847222221898846 0.12847222221898846
 0.11458333334303461 0.11458333334303461 0.11458333334303461
 0.11458333334303461 0.11458333334303461 0.10069444443797693
 0.10069444443797693 0.10069444443797693 0.10069444443797693
 0.10069444443797693 0.08888888888759539 0.08819444445543922
 0.08819444445543922 0.08819444445543922 0.08680555556202307
 0.08680555556202307 0.08680555556202307 0.08680555556202307
 0.08680555556202307 0.07291666665696539 0.07291666665696539
 0.07291666665696539 0.07291666665696539 0.07291666665696539
 0.05902777778101154 0.05902777778101154 0.05902777778101154
 0.05902777778101154 0.05902777778101154 0.047222222230629995
 0.046527777769370005 0.046527777769370005 0.046527777769370005
 0.04513888887595385 0.04513888887595385 0.04513888887595385
 0.04513888887595385 0.04513888887595385 0.03125 0.03125 0.03125 0.

In [12]:
print(dt.isoformat()[14:16])

00


In [13]:
print(len(obs_s))

56
